In [1]:
#Step to download the yolov6 o
%cd /kaggle/working
!git clone https://github.com/ultralytics/yolov5

/kaggle/working
Cloning into 'yolov5'...
remote: Enumerating objects: 17851, done.
remote: Counting objects: 100% (53/53), done.
remote: Compressing objects: 100% (44/44), done.
remote: Total 17851 (delta 33), reused 9 (delta 9), pack-reused 17798 (from 2)
Receiving objects: 100% (17851/17851), 17.00 MiB | 20.60 MiB/s, done.
Resolving deltas: 100% (12169/12169), done.


In [ ]:
%cd /kaggle/working/yolov5
!pip install -r requirements.txt
!pip uninstall wandb -qy  # deprecated dependency
import torch

from IPython.display import Image, clear_output 

/kaggle/working/yolov5
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 24.5 MB/s eta 0:00:0000:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 131.6/131.6 kB 8.1 MB/s eta 0:00:00
  Attempting uninstall: urllib3
    Found existing installation: urllib3 2.5.0
    Uninstalling urllib3-2.5.0:
      Successfully uninstalled urllib3-2.5.0
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-adk 1.25.1 requires google-cloud-bigquery-storage>=2.0.0, which is not installed.
gcsfs 2025.3.0 requires fsspec==2025.3.0, but you have fsspec 2026.2.0 which is incompatible.


In [ ]:
!pip install roboflow

In [ ]:
# Step2 : to download the dataset
import os
from roboflow import Roboflow

ROOT = os.getcwd()
DATASETS_DIR = os.path.join(ROOT, 'datasets')
os.makedirs(DATASETS_DIR, exist_ok=True)
os.chdir(DATASETS_DIR)

rf = Roboflow(api_key="VjWiwna9c7fEdneXcjw6")
print("Downloading Dataset 1... (ziptol)")
rf.workspace("tablet-ab3ji").project("ziptol").version(2).download("yolov5")


print("Downloading Dataset 2.. (Coin detection)") 
rf.workspace("anazs-workspace").project("indian-currency-coin-detection-97vpl").version(1).download("yolov5")
# print("Downloading Dataset 3... (Detect Indian Currency)")
# project = rf.workspace("iit-pallakkad").project("detect-indian-currency")
# version = project.version(1)
# dataset = version.download("yolov5")
os.chdir(ROOT)
print("✅ All Downloads Complete inside /datasets folder.")



In [ ]:
#Step4: converting bounding box to segmentation
!wget -q https://github.com/ultralytics/assets/releases/download/v8.2.0/sam_b.pt

In [ ]:
#step conversion 
def convert_dataset_to_segmentation(dataset_path, sam_model_path="/kaggle/working/yolov5/sam_b.pt"):
    sam_model = SAM(sam_model_path)

    for split in ['train', 'valid', 'test']:
        print(f"\n🚀 Processing {split}...")

        img_dir = Path(dataset_path) / split / "images"
        lbl_dir = Path(dataset_path) / split / "labels"

        if not img_dir.exists():
            continue

        dataset = YOLODataset(str(img_dir), data=dict(names=list(range(1000))))

        for l in tqdm(dataset.labels, desc=f"{split} conversion"):
            h, w = l["shape"]
            boxes = l["bboxes"]

            if len(boxes) == 0:
                continue

            # Convert normalized → pixel coords
            boxes[:, [0, 2]] *= w
            boxes[:, [1, 3]] *= h

            im = cv2.imread(l["im_file"])

            sam_results = sam_model(
                im,
                bboxes=xywh2xyxy(boxes),
                verbose=False
            )

            segments = sam_results[0].masks.xyn
            cls = l["cls"]

            txt_path = lbl_dir / Path(l["im_file"]).with_suffix(".txt").name

            new_lines = []

            for i, seg in enumerate(segments):
                if seg is None or len(seg) < 3:
                    continue

                coords = seg.reshape(-1)

                # ✅ Clamp values
                coords = [max(0.0, min(1.0, float(c))) for c in coords]

                line = f"{int(cls[i])} " + " ".join([f"{c:.6f}" for c in coords])
                new_lines.append(line)

            # ❌ Remove empty labels
            if not new_lines:
                img_file = Path(l["im_file"])
                if img_file.exists():
                    img_file.unlink()
                if txt_path.exists():
                    txt_path.unlink()
                continue

            # ✅ Overwrite label
            with open(txt_path, "w") as f:
                f.write("\n".join(new_lines))

    print("\n✅ Conversion complete (labels replaced with segmentation)")

In [ ]:
from roboflow import Roboflow
from ultralytics import SAM
from ultralytics.data import YOLODataset
from ultralytics.utils.ops import xywh2xyxy
from ultralytics.utils import LOGGER
from pathlib import Path
import cv2
import shutil
from tqdm import tqdm

# Ensure SAM is imported, as it caused a NameError previously.
# Then call the conversion function with the corrected dataset path.
convert_dataset_to_segmentation("/kaggle/working/yolov5/datasets/Indian-Currency-&-Coin-detection-1", "/kaggle/working/yolov5/sam_b.pt")

In [ ]:
import os
import shutil
import yaml
from pathlib import Path
from tqdm import tqdm

# ================= CONFIG =================
DATASETS_ROOT = "datasets"
OUTPUT_DIR = "merged_dataset"
UNIFIED_NAMES = [
    '10Rupee_note', '20Rupee_note', '50Rupee_note', '100Rupee_note', 
    '200Rupee_note', '500Rupee_note', '2000Rupee_note', '5Rupee_note', 
    '1Rupee_coin', '2Rupee_coin', '5Rupee_coin', '10Rupee_coin', 'None'
]

MAPPINGS = {
    'ziptol-2': {'n10': '10Rupee_note', 'n20': '20Rupee_note', 'n50': '50Rupee_note', 'n100': '100Rupee_note', 'n200': '200Rupee_note', 'n500': '500Rupee_note'},
    
   # 'Detect-Indian-Currency-1': {
   #      '10Rupee_note': '10Rupee_note', '20Rupee_note': '20Rupee_note', '50Rupee_note': '50Rupee_note', 
   #      '100Rupee_note': '100Rupee_note', '200Rupee_note': '200Rupee_note', '500Rupee_note': '500Rupee_note', 
   #      '2000Rupee_note': '2000Rupee_note', '1Rupee_coin': '1Rupee_coin', '2Rupee_coin': '2Rupee_coin', 
   #      '5Rupee_coin': '5Rupee_coin', '10Rupee_coin': '10Rupee_coin', 'undefined': 'None'
   #  },
    'Indian-Currency-&-Coin-detection-1': {
        '10': '10Rupee_note', '20': '20Rupee_note', '50': '50Rupee_note', '100': '100Rupee_note', 
        '200': '200Rupee_note', '500': '500Rupee_note', '2000': '2000Rupee_note', 
        '1': '1Rupee_coin', '2': '2Rupee_coin', '5': '5Rupee_coin', 
        '10_coin_ref': '10Rupee_coin'
    }
    
}

# ==========================================
def merge_datasets():
    if os.path.exists(OUTPUT_DIR): shutil.rmtree(OUTPUT_DIR)
    for split in ['train', 'valid', 'test']:
        os.makedirs(f"{OUTPUT_DIR}/{split}/images", exist_ok=True)
        os.makedirs(f"{OUTPUT_DIR}/{split}/labels", exist_ok=True)

    total_processed = 0
    sanitized_count = 0
    
    for ds_id, ds_name in enumerate(MAPPINGS.keys()):
        print(f"Processing {ds_name}...")
        ds_path = Path(DATASETS_ROOT) / ds_name
        yaml_path = ds_path / 'data.yaml'
        if not yaml_path.exists():
            print(f"  ⚠️ Skipping {ds_name} (data.yaml not found)")
            continue
        
        with open(yaml_path, 'r') as f: 
            orig_names = yaml.safe_load(f)['names']
            
        local_id_map = {}
        for i, old_name in enumerate(orig_names):
            target_name = MAPPINGS[ds_name].get(old_name, 'None')
            local_id_map[i] = UNIFIED_NAMES.index(target_name) if target_name in UNIFIED_NAMES else UNIFIED_NAMES.index('None')

        for split in ['train', 'valid', 'test']:
            src_img_dir = ds_path / split / 'images'
            if not src_img_dir.exists() and split == 'valid': src_img_dir = ds_path / 'val' / 'images'
            if not src_img_dir.exists(): continue
            
            src_lbl_dir = Path(str(src_img_dir).replace('images', 'labels'))
            img_files = list(src_img_dir.glob('*'))
            
            for img in tqdm(img_files, desc=f"  {split}", leave=False):
                new_name = f"ds{ds_id}_{img.name}"
                src_lbl = src_lbl_dir / (img.stem + ".txt")
                if not src_lbl.exists(): continue
                
                shutil.copy(img, Path(OUTPUT_DIR) / split / 'images' / new_name)
                with open(src_lbl, 'r') as f_in, open(Path(OUTPUT_DIR) / split / 'labels' / new_name.replace(img.suffix, '.txt'), 'w') as f_out:
                    for line in f_in:
                        parts = line.strip().split()
                        if not parts: continue
                        new_cid = local_id_map.get(int(parts[0]), UNIFIED_NAMES.index('None'))
                        coords = []
                        for x in parts[1:]:
                            f_val = float(x)
                            if f_val < 0.0 or f_val > 1.0: sanitized_count += 1
                            coords.append(max(0.0, min(1.0, f_val)))
                        f_out.write(f"{new_cid} {' '.join([f'{c:.6f}' for c in coords])}\n")
                total_processed += 1

    yaml_data = {'train': os.path.join(OUTPUT_DIR, 'train', 'images'), 'val': os.path.join(OUTPUT_DIR, 'valid', 'images'), 'test': os.path.join(OUTPUT_DIR, 'test', 'images'), 'nc': len(UNIFIED_NAMES), 'names': UNIFIED_NAMES}
    with open(os.path.join(OUTPUT_DIR, 'data.yaml'), 'w') as f: yaml.dump(yaml_data, f)
    print(f"\n✅ Master Merge Complete. {total_processed} processed, {sanitized_count} sanitized.")

merge_datasets()
if os.path.exists(DATASETS_ROOT):
    shutil.rmtree(DATASETS_ROOT)
    print(f"✅ Deleted {DATASETS_ROOT} to free up space.")

       
            
                        


In [ ]:
import os
import yaml
from pathlib import Path
from collections import Counter
from tqdm import tqdm

MERGED_DIR ="/kaggle/working/yolov5/merged_dataset"

def deep_audit():
    print("--- Starting Deep Audit of Merged  Dataset ---")
    yaml_path = os.path.join(MERGED_DIR, 'data.yaml')
    if not os.path.exists(yaml_path):
        print("❌ Error: data.yaml not found!")
        return

    with open(yaml_path, 'r') as f:
        data = yaml.safe_load(f)
        nc = data['nc']
        names = data['names']

    print(f"Dataset Name: ViewSense 75k Master")
    print(f"Expected Classes: {nc} {names}")

    total_images = 0
    total_labels = 0
    out_of_range_coords = 0
    invalid_class_ids = 0
    distribution = Counter()

    for split in ['train', 'valid', 'test']:
        img_dir = Path(MERGED_DIR) / split / 'images'
        lbl_dir = Path(MERGED_DIR) / split / 'labels'
        
        if not img_dir.exists():
            print(f"⚠️ Warning: Split {split} images not found.")
            continue

        images = list(img_dir.glob('*'))
        total_images += len(images)
        
        print(f"Auditing {split} split ({len(images)} images)...")
        
        for img_p in tqdm(images, desc=f"Verifying {split}", leave=False):
            lbl_p = lbl_dir / (img_p.stem + ".txt")
            if not lbl_p.exists():
                continue
            
            total_labels += 1
            with open(lbl_p, 'r') as f:
                for line in f:
                    parts = line.strip().split()
                    if not parts: continue
                    
                    # 1. Class ID Verification
                    cid = int(parts[0])
                    if cid < 0 or cid >= nc:
                        invalid_class_ids += 1
                    distribution[cid] += 1
                    
                    # 2. Coordinate Range Verification
                    for coord in parts[1:]:
                        val = float(coord)
                        if val < 0.0 or val > 1.0:
                            out_of_range_coords += 1

    print("\n" + "="*40)
    print(" AUDIT RESULTS")
    print("="*40)
    print(f"Total Images on Disk: {total_images}")
    print(f"Total Labels on Disk: {total_labels}")
    print(f"Invalid Class IDs (Should be 0): {invalid_class_ids}")
    print(f"Out-of-range Coordinates (Should be 0): {out_of_range_coords}")
    
    print("\nVerified Class Distribution:")
    for i, name in enumerate(names):
        print(f" {i}: {name.ljust(15)} -> {distribution[i]} instances")

    if out_of_range_coords == 0 and invalid_class_ids == 0:
        print("\n✅ VERDICT: Dataset is 100% Valid and Mathematically Sanitized.")
    else:
        print("\n❌ VERDICT: Dataset has inconsistencies. Re-run merge.")

if __name__ == "__main__":
    deep_audit()


In [ ]:
%cd /kaggle/working/yolov5
!python segment/train.py \
  --img 320 \
  --batch 16 \
  --epochs 100 \
  --data /kaggle/working/yolov5/merged_dataset/data.yaml\
   --weights yolov5s-seg.pt\
  --name yolo-currency \
  --cache \
  --device 0
  
 